In [ ]:
# Example 1: Basic system information and process monitoring
import psutil
import os

print("Example 1: Basic system information and process monitoring")

# System-wide information
print(f"CPU count (logical): {psutil.cpu_count()}")
print(f"CPU count (physical): {psutil.cpu_count(logical=False)}")
print(f"CPU usage: {psutil.cpu_percent(interval=1)}%")

# Memory information
memory = psutil.virtual_memory()
print(f"Memory total: {memory.total / (1024**3):.2f} GB")
print(f"Memory available: {memory.available / (1024**3):.2f} GB")
print(f"Memory used: {memory.percent}%")

# Current process information
current_process = psutil.Process()
print(f"\nCurrent process PID: {current_process.pid}")
print(f"Process name: {current_process.name()}")
print(f"Process status: {current_process.status()}")
print(f"Process CPU usage: {current_process.cpu_percent()}%")
print(f"Process memory usage: {current_process.memory_info().rss / (1024**2):.2f} MB")

# System boot time
import datetime
boot_time = datetime.datetime.fromtimestamp(psutil.boot_time())
print(f"\nSystem boot time: {boot_time}")

print("Basic system monitoring completed successfully!")

In [ ]:
# Example 2: Disk and file system monitoring
import psutil

print("Example 2: Disk and file system monitoring")

# Get all disk partitions
partitions = psutil.disk_partitions()
print(f"Found {len(partitions)} disk partitions:")

for partition in partitions:
    print(f"\nPartition: {partition.device}")
    print(f"  Mountpoint: {partition.mountpoint}")
    print(f"  File system: {partition.fstype}")
    
    try:
        # Get disk usage for this partition
        usage = psutil.disk_usage(partition.mountpoint)
        print(f"  Total size: {usage.total / (1024**3):.2f} GB")
        print(f"  Used: {usage.used / (1024**3):.2f} GB")
        print(f"  Free: {usage.free / (1024**3):.2f} GB")
        print(f"  Usage: {(usage.used / usage.total) * 100:.1f}%")
    except PermissionError:
        print(f"  Permission denied to access {partition.mountpoint}")

# System-wide disk I/O statistics
print(f"\n--- Disk I/O Statistics ---")
disk_io = psutil.disk_io_counters()
if disk_io:
    print(f"Read count: {disk_io.read_count}")
    print(f"Write count: {disk_io.write_count}")
    print(f"Read bytes: {disk_io.read_bytes / (1024**2):.2f} MB")
    print(f"Write bytes: {disk_io.write_bytes / (1024**2):.2f} MB")
    print(f"Read time: {disk_io.read_time} ms")
    print(f"Write time: {disk_io.write_time} ms")
else:
    print("Disk I/O statistics not available on this system")

# Per-disk I/O statistics
print(f"\n--- Per-Disk I/O Statistics ---")
per_disk_io = psutil.disk_io_counters(perdisk=True)
for disk, stats in list(per_disk_io.items())[:3]:  # Show first 3 disks
    print(f"Disk {disk}:")
    print(f"  Reads: {stats.read_count} ({stats.read_bytes / (1024**2):.2f} MB)")
    print(f"  Writes: {stats.write_count} ({stats.write_bytes / (1024**2):.2f} MB)")

print("Disk monitoring completed successfully!")

In [ ]:
# Example 3: Network interface and connection monitoring
import psutil
from collections import defaultdict

print("Example 3: Network interface and connection monitoring")

# Network interfaces and their addresses
print("--- Network Interfaces ---")
interfaces = psutil.net_if_addrs()
for interface_name, addresses in interfaces.items():
    print(f"\nInterface: {interface_name}")
    for addr in addresses:
        print(f"  {addr.family.name}: {addr.address}")
        if addr.netmask:
            print(f"    Netmask: {addr.netmask}")
        if addr.broadcast:
            print(f"    Broadcast: {addr.broadcast}")

# Network interface statistics
print(f"\n--- Network Interface Statistics ---")
net_stats = psutil.net_if_stats()
for interface_name, stats in list(net_stats.items())[:3]:  # Show first 3
    print(f"\n{interface_name}:")
    print(f"  Up: {stats.isup}")
    print(f"  Duplex: {stats.duplex}")
    print(f"  Speed: {stats.speed} Mbps")
    print(f"  MTU: {stats.mtu}")

# Network I/O statistics
print(f"\n--- Network I/O Statistics ---")
net_io = psutil.net_io_counters()
print(f"Total bytes sent: {net_io.bytes_sent / (1024**2):.2f} MB")
print(f"Total bytes received: {net_io.bytes_recv / (1024**2):.2f} MB")
print(f"Packets sent: {net_io.packets_sent}")
print(f"Packets received: {net_io.packets_recv}")
print(f"Errors in: {net_io.errin}, Errors out: {net_io.errout}")
print(f"Drops in: {net_io.dropin}, Drops out: {net_io.dropout}")

# Per-interface I/O statistics
print(f"\n--- Per-Interface I/O Statistics ---")
per_nic_io = psutil.net_io_counters(pernic=True)
for interface, stats in list(per_nic_io.items())[:2]:  # Show first 2
    print(f"\n{interface}:")
    print(f"  Sent: {stats.bytes_sent / (1024**2):.2f} MB ({stats.packets_sent} packets)")
    print(f"  Received: {stats.bytes_recv / (1024**2):.2f} MB ({stats.packets_recv} packets)")

# Network connections - try to get connections, handle permission errors
print(f"\n--- Active Network Connections ---")
try:
    # On macOS/Unix, this may require elevated privileges
    connections = psutil.net_connections(kind='inet')[:5]
    for i, conn in enumerate(connections, 1):
        print(f"Connection {i}:")
        print(f"  Protocol: {conn.type.name}")
        print(f"  Local: {conn.laddr}")
        print(f"  Remote: {conn.raddr}")
        print(f"  Status: {conn.status}")
        print(f"  PID: {conn.pid}")
except psutil.AccessDenied:
    print("Access denied to network connections (requires elevated privileges on macOS)")
    print("Showing current process connections instead:")
    # Get connections for current process only
    current_proc = psutil.Process()
    try:
        proc_connections = current_proc.net_connections(kind='inet')[:5]
        if proc_connections:
            for i, conn in enumerate(proc_connections, 1):
                print(f"Connection {i}:")
                print(f"  Protocol: {conn.type.name}")
                print(f"  Local: {conn.laddr}")
                if conn.raddr:
                    print(f"  Remote: {conn.raddr}")
                print(f"  Status: {conn.status}")
        else:
            print("No active connections for current process")
    except psutil.AccessDenied:
        print("Unable to access connection information")

print("Network monitoring completed successfully!")

In [ ]:
# Example 4: Process management and control
import psutil
import os
import time
from collections import defaultdict

print("Example 4: Process management and control")

# Find processes by name
print("--- Process Discovery ---")
python_processes = []
for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
    try:
        if 'python' in proc.info['name'].lower():
            python_processes.append(proc)
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        pass

print(f"Found {len(python_processes)} Python processes:")
for proc in python_processes[:3]:  # Show first 3
    try:
        print(f"  PID {proc.pid}: {proc.name()} - CPU: {proc.cpu_percent():.1f}%")
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        print(f"  PID {proc.pid}: Process no longer accessible")

# Process tree analysis
print(f"\n--- Process Tree Analysis ---")
current_proc = psutil.Process()
print(f"Current process: {current_proc.name()} (PID: {current_proc.pid})")

# Get parent process
try:
    parent = current_proc.parent()
    if parent:
        print(f"Parent process: {parent.name()} (PID: {parent.pid})")
        print(f"  Parent CPU usage: {parent.cpu_percent():.1f}%")
        print(f"  Parent memory: {parent.memory_info().rss / (1024**2):.2f} MB")
except (psutil.NoSuchProcess, psutil.AccessDenied):
    print("Parent process not accessible")

# Get child processes
children = current_proc.children()
print(f"Child processes: {len(children)}")
for child in children[:2]:  # Show first 2
    try:
        print(f"  Child PID {child.pid}: {child.name()}")
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        pass

# Top processes by CPU usage
print(f"\n--- Top Processes by CPU Usage ---")
processes = []
for proc in psutil.process_iter(['pid', 'name', 'cpu_percent']):
    try:
        proc.cpu_percent()  # Call once to initialize
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        pass

time.sleep(1)  # Wait for CPU percentage calculation

cpu_processes = []
for proc in psutil.process_iter(['pid', 'name', 'cpu_percent']):
    try:
        cpu_usage = proc.cpu_percent()
        if cpu_usage > 0:
            cpu_processes.append((proc.pid, proc.name(), cpu_usage))
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        pass

# Sort by CPU usage and show top 5
cpu_processes.sort(key=lambda x: x[2], reverse=True)
for pid, name, cpu in cpu_processes[:5]:
    print(f"  PID {pid}: {name} - {cpu:.1f}% CPU")

# Process status distribution
print(f"\n--- Process Status Distribution ---")
status_count = defaultdict(int)
for proc in psutil.process_iter(['status']):
    try:
        status_count[proc.info['status']] += 1
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        pass

for status, count in sorted(status_count.items()):
    print(f"  {status}: {count} processes")

print("Process management completed successfully!")

In [ ]:
# Example 5: System performance monitoring and alerting
import psutil
import time
from datetime import datetime

print("Example 5: System performance monitoring and alerting")

class SystemMonitor:
    """System performance monitor with alerting capabilities"""
    
    def __init__(self, cpu_threshold=80, memory_threshold=85, disk_threshold=90):
        self.cpu_threshold = cpu_threshold
        self.memory_threshold = memory_threshold  
        self.disk_threshold = disk_threshold
        self.alerts = []
    
    def check_cpu_usage(self):
        """Monitor CPU usage across multiple cores"""
        # Overall CPU usage
        overall_cpu = psutil.cpu_percent(interval=1)
        
        # Per-core CPU usage
        per_core_cpu = psutil.cpu_percent(percpu=True, interval=1)
        
        cpu_info = {
            'overall': overall_cpu,
            'per_core': per_core_cpu,
            'timestamp': datetime.now(),
            'alert': overall_cpu > self.cpu_threshold
        }
        
        if cpu_info['alert']:
            self.alerts.append(f"HIGH CPU: {overall_cpu:.1f}% (threshold: {self.cpu_threshold}%)")
        
        return cpu_info
    
    def check_memory_usage(self):
        """Monitor memory usage including virtual and swap memory"""
        virtual_mem = psutil.virtual_memory()
        swap_mem = psutil.swap_memory()
        
        memory_info = {
            'virtual': {
                'total': virtual_mem.total,
                'available': virtual_mem.available,
                'percent': virtual_mem.percent,
                'used': virtual_mem.used,
                'free': virtual_mem.free
            },
            'swap': {
                'total': swap_mem.total,
                'used': swap_mem.used,
                'free': swap_mem.free,
                'percent': swap_mem.percent
            },
            'timestamp': datetime.now(),
            'alert': virtual_mem.percent > self.memory_threshold
        }
        
        if memory_info['alert']:
            self.alerts.append(f"HIGH MEMORY: {virtual_mem.percent:.1f}% (threshold: {self.memory_threshold}%)")
        
        return memory_info
    
    def check_disk_usage(self):
        """Monitor disk usage across all mounted partitions"""
        disk_info = {}
        
        for partition in psutil.disk_partitions():
            try:
                usage = psutil.disk_usage(partition.mountpoint)
                usage_percent = (usage.used / usage.total) * 100
                
                disk_info[partition.device] = {
                    'mountpoint': partition.mountpoint,
                    'total': usage.total,
                    'used': usage.used,
                    'free': usage.free,
                    'percent': usage_percent,
                    'alert': usage_percent > self.disk_threshold
                }
                
                if usage_percent > self.disk_threshold:
                    self.alerts.append(f"HIGH DISK: {partition.device} {usage_percent:.1f}% (threshold: {self.disk_threshold}%)")
                    
            except PermissionError:
                disk_info[partition.device] = {'error': 'Permission denied'}
        
        return disk_info
    
    def get_system_health_report(self):
        """Generate comprehensive system health report"""
        print("=== System Health Report ===")
        print(f"Report generated at: {datetime.now()}")
        
        # CPU monitoring
        print(f"\n--- CPU Status ---")
        cpu_info = self.check_cpu_usage()
        print(f"Overall CPU usage: {cpu_info['overall']:.1f}%")
        # Format per-core usage separately to avoid nested f-string issues
        core_usage = [f"{cpu:.1f}%" for cpu in cpu_info['per_core'][:4]]
        print(f"Per-core usage: {core_usage}")  # Show first 4 cores
        
        # CPU frequency (if available)
        try:
            cpu_freq = psutil.cpu_freq()
            print(f"CPU frequency: Current={cpu_freq.current:.0f}MHz, Min={cpu_freq.min:.0f}MHz, Max={cpu_freq.max:.0f}MHz")
        except:
            print("CPU frequency information not available")
        
        # Memory monitoring
        print(f"\n--- Memory Status ---")
        memory_info = self.check_memory_usage()
        vm = memory_info['virtual']
        print(f"Virtual memory: {vm['percent']:.1f}% used ({vm['used']/(1024**3):.2f}GB / {vm['total']/(1024**3):.2f}GB)")
        print(f"Available memory: {vm['available']/(1024**3):.2f}GB")
        
        if memory_info['swap']['total'] > 0:
            sw = memory_info['swap']
            print(f"Swap memory: {sw['percent']:.1f}% used ({sw['used']/(1024**3):.2f}GB / {sw['total']/(1024**3):.2f}GB)")
        
        # Disk monitoring
        print(f"\n--- Disk Status ---")
        disk_info = self.check_disk_usage()
        for device, info in list(disk_info.items())[:3]:  # Show first 3 disks
            if 'error' not in info:
                print(f"{device}: {info['percent']:.1f}% used ({info['used']/(1024**3):.2f}GB / {info['total']/(1024**3):.2f}GB)")
            else:
                print(f"{device}: {info['error']}")
        
        # System load average (Unix-like systems)
        try:
            load_avg = psutil.getloadavg()
            print(f"\nLoad average: {load_avg[0]:.2f}, {load_avg[1]:.2f}, {load_avg[2]:.2f} (1, 5, 15 min)")
        except AttributeError:
            print("\nLoad average not available on this platform")
        
        # Process count
        process_count = len(psutil.pids())
        print(f"Running processes: {process_count}")
        
        # Alerts summary
        if self.alerts:
            print(f"\n--- ALERTS ({len(self.alerts)}) ---")
            for alert in self.alerts[-5:]:  # Show last 5 alerts
                print(f"⚠️  {alert}")
        else:
            print(f"\n--- No alerts - System operating normally ---")

# Initialize system monitor
monitor = SystemMonitor(cpu_threshold=70, memory_threshold=80, disk_threshold=85)

# Generate system health report
monitor.get_system_health_report()

# Simulate continuous monitoring (brief example)
print(f"\n--- Continuous Monitoring Sample (3 seconds) ---")
for i in range(3):
    time.sleep(1)
    cpu = psutil.cpu_percent(interval=0.1)
    memory = psutil.virtual_memory().percent
    print(f"Second {i+1}: CPU={cpu:.1f}%, Memory={memory:.1f}%")

print("System performance monitoring completed successfully!")